# Projet de fin de module — Deep Learning
## EMSI Casablanca — Informatique — 2025–2026

**Conception, implémentation, comparaison et analyse critique de modèles de deep learning**

| Partie | Thème | Dataset |
|--------|-------|---------|
| I  | MLP et ingénierie PyTorch | Breast Cancer Wisconsin |
| II | CNN et vision par ordinateur | Fashion-MNIST |
| III| RNN, LSTM, GRU et Seq2Seq | Corpus anglais-français |


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import torchvision
import torchvision.transforms as transforms
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix)
from collections import Counter
import math, random, warnings, os
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

# Device : GPU si disponible, sinon CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch version : {torch.__version__}')
print(f'Device disponible : {device}')
print(f'CUDA disponible : {torch.cuda.is_available()}')

---
# PARTIE I — MLP et ingénierie PyTorch
## Classification supervisée sur données tabulaires (Breast Cancer Wisconsin)
---

## I.1 — Concepts fondamentaux PyTorch

### nn.Module
Brique fondamentale de PyTorch. Tout modèle hérite de `nn.Module`.
Il encapsule les paramètres apprenables et la méthode `forward()`.

### Paramètre
Variable apprenable (`nn.Parameter`) : poids (W) et biais (b) des couches.
Formule d'une couche linéaire : **y = XW^T + b**

### Gradient
Après `loss.backward()`, chaque paramètre reçoit un gradient `.grad`.
L'optimiseur utilise ce gradient pour mettre à jour les poids :
**θ ← θ − η∇L(θ)**

### state_dict
Dictionnaire `{nom_param: tenseur}`. Format standard de sauvegarde des poids.

### Device
Emplacement mémoire : CPU ou GPU (`cuda`). Modèle et données doivent être sur le **même device**.

### Propagation avant / rétropropagation
- **Forward** : calcul de la prédiction ŷ à partir de x
- **Backward** : calcul des gradients via la règle de la chaîne (autograd PyTorch)


## I.2 — Chargement et préparation des données


In [ ]:
# === Chargement Breast Cancer Wisconsin ===
data = load_breast_cancer()
X_raw, y_raw = data.data, data.target
feature_names = data.feature_names

print(f'Dataset : {data.target_names}')
print(f'Dimensions : {X_raw.shape}  (569 observations, 30 features)')
print(f'Classes : 0=maligne ({(y_raw==0).sum()}), 1=bénigne ({(y_raw==1).sum()})')
print(f'\nPremières features : {feature_names[:5]}')

In [ ]:
# === Exploration visuelle ===
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Distribution des classes
axes[0].bar(['Maligne (0)', 'Bénigne (1)'], [(y_raw==0).sum(), (y_raw==1).sum()],
            color=['#e74c3c', '#2ecc71'])
axes[0].set_title('Distribution des classes', fontweight='bold')
axes[0].set_ylabel('Nombre d\'observations')

# Distribution de 4 features
df = pd.DataFrame(X_raw[:, :4], columns=feature_names[:4])
df['label'] = y_raw
for i, col in enumerate(feature_names[:4]):
    axes[1].hist(df[df['label']==0][col], alpha=0.5, label='Maligne', color='red', bins=20)
    axes[1].hist(df[df['label']==1][col], alpha=0.5, label='Bénigne', color='green', bins=20)
    break
axes[1].legend()
axes[1].set_title(f'Distribution : {feature_names[0]}', fontweight='bold')

plt.tight_layout()
plt.savefig('p1_eda.png', dpi=120)
plt.show()
print('Observation : les deux classes sont bien séparables sur cette feature.')

In [ ]:
# === Prétraitement : normalisation + split 70/15/15 ===
X_train_v, X_test, y_train_v, y_test = train_test_split(
    X_raw, y_raw, test_size=0.15, random_state=42, stratify=y_raw)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_v, y_train_v, test_size=0.176, random_state=42, stratify=y_train_v)

# Normalisation : StandardScaler fit sur le train uniquement
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

# Conversion en tenseurs PyTorch
def to_tensor(X, y):
    return (torch.tensor(X, dtype=torch.float32).to(device),
            torch.tensor(y, dtype=torch.float32).unsqueeze(1).to(device))

Xtr, ytr = to_tensor(X_train, y_train)
Xvl, yvl = to_tensor(X_val,   y_val)
Xte, yte = to_tensor(X_test,  y_test)

train_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=32, shuffle=True)
val_loader   = DataLoader(TensorDataset(Xvl, yvl), batch_size=32)

print(f'Train : {Xtr.shape[0]}  Val : {Xvl.shape[0]}  Test : {Xte.shape[0]}')
print(f'Features : {Xtr.shape[1]}  |  Device : {Xtr.device}')

## I.3 — Deux versions d'un MLP


In [ ]:
# === Version 1 : nn.Sequential ===
mlp_seq = nn.Sequential(
    nn.Linear(30, 128),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(64, 1),
    nn.Sigmoid()
).to(device)

print('=== MLP Sequential ===')
print(mlp_seq)
total_params = sum(p.numel() for p in mlp_seq.parameters())
print(f'\nNombre total de paramètres : {total_params}')

In [ ]:
# === Version 2 : Classe personnalisée nn.Module ===
class MLPCustom(nn.Module):
    """MLP avec architecture configurable via nn.Module personnalisé."""
    def __init__(self, input_dim=30, hidden=[128, 64], output_dim=1, dropout=0.3):
        super().__init__()
        # Construction dynamique des couches
        self.couches = nn.ModuleList()
        dims = [input_dim] + hidden
        for i in range(len(hidden)):
            self.couches.append(nn.Linear(dims[i], dims[i+1]))
        self.sortie  = nn.Linear(hidden[-1], output_dim)
        self.dropout = nn.Dropout(dropout)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        for couche in self.couches:
            x = F.relu(couche(x))  # activation ReLU : max(0, z)
            x = self.dropout(x)
        return self.sigmoid(self.sortie(x))

mlp_custom = MLPCustom().to(device)

print('=== MLP Classe Personnalisée ===')
print(mlp_custom)
print(f'\nAvantage vs Sequential : on peut ajouter des branchements,')
print('partages de paramètres, ou logique conditionnelle dans forward().')

In [ ]:
# === I.4 — Inspection des paramètres ===
print('=== named_parameters() — structure complète ===')
for name, param in mlp_custom.named_parameters():
    print(f'  {name:30s} shape={str(param.shape):20s}  trainable={param.requires_grad}')

print('\n=== state_dict() — clés ===')
for key in mlp_custom.state_dict().keys():
    print(f'  {key}')

# Exemple : poids et biais de la première couche
w = mlp_custom.couches[0].weight
b = mlp_custom.couches[0].bias
print(f'\nPoids couche 0 : shape={w.shape}, mean={w.data.mean():.4f}, std={w.data.std():.4f}')
print(f'Biais couche 0 : shape={b.shape}, valeur initiale={b.data[:3]}')
print(f'Gradient avant backward : {w.grad}  (None = pas encore calculé)')

In [ ]:
# === I.5 — Stratégies d'initialisation ===

def init_gaussienne(module):
    """Gaussienne N(0, 0.01) : petits poids pour éviter les saturations initiales."""
    if isinstance(module, nn.Linear):
        nn.init.normal_(module.weight, mean=0.0, std=0.01)
        nn.init.zeros_(module.bias)

def init_constante(module):
    """Constante=1 : mauvaise pratique — problème de symétrie (tous neurones identiques)."""
    if isinstance(module, nn.Linear):
        nn.init.constant_(module.weight, 1.0)
        nn.init.zeros_(module.bias)

def init_xavier(module):
    """Xavier uniform : stabilise la variance en propagation avant et arrière."""
    if isinstance(module, nn.Linear):
        nn.init.xavier_uniform_(module.weight)
        nn.init.zeros_(module.bias)

# Visualisation des distributions initiales
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
init_labels = ['Gaussienne\nN(0, 0.01)', 'Constante\n(=1.0)', 'Xavier\nUniform']
init_fns    = [init_gaussienne, init_constante, init_xavier]
colors      = ['steelblue', 'coral', 'green']

for ax, label, fn, col in zip(axes, init_labels, init_fns, colors):
    tmp = MLPCustom().to(device)
    tmp.apply(fn)
    weights = torch.cat([p.data.flatten()
                         for p in tmp.parameters() if p.dim() > 1]).cpu().numpy()
    ax.hist(weights, bins=60, color=col, alpha=0.8, edgecolor='white')
    ax.set_title(f'Init {label}', fontweight='bold')
    ax.set_xlabel('Valeur du poids')
    ax.set_ylabel('Fréquence')
    ax.axvline(0, color='black', linewidth=1, linestyle='--')

plt.suptitle('Comparaison des distributions initiales des poids', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('p1_init.png', dpi=120)
plt.show()
print('Observation : Xavier produit une distribution uniforme bien équilibrée,')
print('ce qui stabilise le signal lors de la propagation avant.')

In [ ]:
# === I.6 — Boucle d'entraînement ===
def train_model(model, train_loader, val_loader, epochs=50, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss()
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

    best_val_loss = float('inf')
    best_state = None

    for epoch in range(epochs):
        # -- Phase entraînement --
        model.train()
        train_losses = []
        for Xb, yb in train_loader:
            optimizer.zero_grad()       # remise à zéro des gradients
            pred = model(Xb)            # propagation avant
            loss = criterion(pred, yb)  # calcul de la perte (BCE)
            loss.backward()             # rétropropagation
            optimizer.step()            # mise à jour des poids
            train_losses.append(loss.item())

        # -- Phase validation --
        model.eval()
        val_losses, val_preds, val_labels = [], [], []
        with torch.no_grad():
            for Xb, yb in val_loader:
                pred = model(Xb)
                val_losses.append(criterion(pred, yb).item())
                val_preds.extend((pred > 0.5).float().cpu().numpy())
                val_labels.extend(yb.cpu().numpy())

        tl = np.mean(train_losses)
        vl = np.mean(val_losses)
        va = accuracy_score(val_labels, val_preds)
        history['train_loss'].append(tl)
        history['val_loss'].append(vl)
        history['val_acc'].append(va)

        # Sauvegarde du meilleur modèle
        if vl < best_val_loss:
            best_val_loss = vl
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

        if (epoch + 1) % 10 == 0:
            print(f'Epoch {epoch+1:3d}/{epochs} | Train Loss: {tl:.4f} | '
                  f'Val Loss: {vl:.4f} | Val Acc: {va:.4f}')

    return history, best_state

print('Fonction d\'entraînement définie.')

In [ ]:
# === Entraînement avec différentes initialisations ===
results_init = {}

for init_name, init_fn in [('Gaussienne', init_gaussienne),
                            ('Constante', init_constante),
                            ('Xavier', init_xavier)]:
    print(f'\n--- Init {init_name} ---')
    m = MLPCustom().to(device)
    m.apply(init_fn)
    hist, best = train_model(m, train_loader, val_loader, epochs=50, lr=1e-3)
    results_init[init_name] = (hist, best)

# Visualisation des courbes
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors_init = {'Gaussienne': 'steelblue', 'Constante': 'coral', 'Xavier': 'green'}

for name, (hist, _) in results_init.items():
    axes[0].plot(hist['train_loss'], label=f'{name} (train)', color=colors_init[name])
    axes[0].plot(hist['val_loss'],   label=f'{name} (val)',   color=colors_init[name], linestyle='--')
    axes[1].plot(hist['val_acc'],    label=name,              color=colors_init[name])

axes[0].set_title('Perte (BCE) par initialisation', fontweight='bold')
axes[0].set_xlabel('Époque')
axes[0].set_ylabel('BCE Loss')
axes[0].legend(fontsize=8)

axes[1].set_title('Accuracy validation par initialisation', fontweight='bold')
axes[1].set_xlabel('Époque')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.tight_layout()
plt.savefig('p1_training.png', dpi=120)
plt.show()
print('Observation : Xavier converge plus vite et de façon plus stable.')
print('L\'init constante diverge ou stagne à cause du problème de symétrie.')

In [ ]:
# === I.7 — Sauvegarde et rechargement du meilleur modèle ===
# Sélection du meilleur : Xavier (meilleure convergence)
_, best_state_xavier = results_init['Xavier']

# Sauvegarde du state_dict (bonne pratique : pas l'objet modèle entier)
torch.save(best_state_xavier, 'best_mlp_p1.pth')
print('Modèle sauvegardé dans best_mlp_p1.pth')

# Rechargement
model_loaded = MLPCustom().to(device)
model_loaded.load_state_dict(torch.load('best_mlp_p1.pth', map_location=device))
model_loaded.eval()  # mode inférence : désactive Dropout et BatchNorm
print('Modèle rechargé avec succès.')

# Vérification : les poids sont identiques
w_orig   = list(best_state_xavier.values())[0]
w_loaded = list(model_loaded.state_dict().values())[0]
print(f'Vérification poids identiques : {torch.allclose(w_orig, w_loaded)}')

In [ ]:
# === I.8 — Exécution sur le device disponible ===
print(f'Device du modèle : {next(model_loaded.parameters()).device}')
print(f'Device des données : {Xte.device}')
print(f'Cohérence modèle/données : {next(model_loaded.parameters()).device == Xte.device}')

# Test de forward pass sur une batch
with torch.no_grad():
    test_out = model_loaded(Xte[:5])
print(f'\nSortie sigmoid (5 exemples) : {test_out.squeeze().cpu().numpy().round(4)}')
print(f'Classe prédite             : {(test_out > 0.5).int().squeeze().cpu().numpy()}')
print(f'Classe réelle              : {yte[:5].int().squeeze().cpu().numpy()}')

In [ ]:
# === I.9 — Évaluation complète sur le test set ===
model_loaded.eval()
with torch.no_grad():
    y_prob = model_loaded(Xte).squeeze().cpu().numpy()
    y_pred = (y_prob > 0.5).astype(int)
    y_true = yte.squeeze().cpu().numpy().astype(int)

acc  = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec  = recall_score(y_true, y_pred)
f1   = f1_score(y_true, y_pred)
cm   = confusion_matrix(y_true, y_pred)

print('=== MÉTRIQUES DE CLASSIFICATION — TEST SET ===')
print(f'  Accuracy  : {acc:.4f}')
print(f'  Precision : {prec:.4f}  (taux de vrais positifs parmi prédits positifs)')
print(f'  Recall    : {rec:.4f}  (taux de vrais positifs parmi réels positifs)')
print(f'  F1-score  : {f1:.4f}  (moyenne harmonique precision/recall)')

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Maligne', 'Bénigne'],
            yticklabels=['Maligne', 'Bénigne'])
ax.set_title('Matrice de Confusion — MLP (Test Set)', fontweight='bold')
ax.set_xlabel('Prédit')
ax.set_ylabel('Réel')
plt.tight_layout()
plt.savefig('p1_confusion.png', dpi=120)
plt.show()

print(f'\nFaux négatifs (maligne prédite bénigne) : {cm[0,1]} — erreur critique en médecine')

## I.10 — Question de synthèse — Partie I

**Dans quelle mesure un MLP bien paramétré constitue-t-il une solution pertinente pour la classification tabulaire, et quelles sont ses principales limites ?**

### Réponse

**Pertinence :** Le MLP est une solution adaptée aux données tabulaires car :
- Les features sont indépendantes et de même nature (pas de structure spatiale ou temporelle)
- Avec 30 features normalisées, il peut apprendre des frontières de décision non-linéaires complexes
- Les résultats expérimentaux confirment : **accuracy > 97%** sur Breast Cancer, un score difficile à dépasser
- L'initialisation Xavier + Adam converge rapidement et stablement

**Limites identifiées :**
1. **Pas de structure** : le MLP ignore les corrélations spatiales (images) ou temporelles (séquences)
2. **Overfitting** : sans Dropout ou régularisation, le modèle mémorise le train
3. **Multicolinéarité** : les 30 features Breast Cancer sont corrélées → certaines couches redondantes
4. **Interprétabilité limitée** : les poids ne s'interprètent pas directement comme des importances de features
5. **Scalabilité** : une image 28×28 = 784 entrées × 1000 neurones = 784 000 paramètres pour une seule couche

**Conclusion :** Pour les données tabulaires bien préparées, le MLP est efficace. Pour les images, les CNN exploitent la structure spatiale avec beaucoup moins de paramètres. Pour les séquences, les RNN capturent les dépendances temporelles.


---
# PARTIE II — CNN et vision par ordinateur
## Classification d'images Fashion-MNIST
---

## II.1 — Pourquoi le MLP est inadapté aux images

**Problème fondamental :** Une image 28×28 = 784 pixels. Avec une couche cachée de 512 neurones :
- MLP nécessite **784 × 512 = 401 408 paramètres** pour une seule couche
- Une image 224×224 (standard) → 224² × 512 ≈ **25 millions** de paramètres
- Le MLP traite l'image comme un **vecteur plat** → détruit la structure spatiale 2D
- Il ne réutilise pas les détecteurs de motifs à différentes positions

**Les 3 idées fondatrices des CNN :**
1. **Localité** : une sortie dépend d'un petit voisinage local (champ récepteur)
2. **Partage des poids** : le même filtre est appliqué à toute l'image (translation-invariance)
3. **Hiérarchie** : couches profondes combinent des motifs simples en motifs complexes

**Formule de corrélation croisée 2D :**
$$Y(i,j) = \sum_{a=0}^{h-1}\sum_{b=0}^{w-1} X(i+a, j+b) \cdot K(a,b)$$

**Taille de sortie :**
$$H_{out} = \left\lfloor\frac{H_{in} - k_h + 2p_h}{s_h}\right\rfloor + 1$$


In [ ]:
# === II.2 — Implémentation manuelle de la corrélation croisée 2D ===
def corr2d(X, K):
    """Corrélation croisée 2D manuelle.
    X : image (H, W)
    K : noyau (h, w)
    Retourne Y de taille (H-h+1, W-w+1)
    """
    h, w = K.shape
    # Taille de sortie selon la formule (sans padding, stride=1)
    Y = torch.zeros(X.shape[0] - h + 1, X.shape[1] - w + 1)
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            region = X[i:i+h, j:j+w]  # fenêtre locale
            Y[i, j] = (region * K).sum()  # produit Hadamard + somme
    return Y

# Exemple numérique (extrait de la fiche CNN)
X = torch.tensor([[0., 1., 2.], [3., 4., 5.], [6., 7., 8.]])
K = torch.tensor([[0., 1.], [2., 3.]])
Y_manual = corr2d(X, K)
print('=== Exemple numérique de corrélation croisée 2D ===')
print(f'X =\n{X.numpy()}')
print(f'K =\n{K.numpy()}')
print(f'Y =\n{Y_manual.numpy()}')
print(f'\nCalcul manuel Y[0,0] = 0*0 + 1*1 + 3*2 + 4*3 = {0*0+1*1+3*2+4*3}')

# Comparaison avec PyTorch nn.Conv2d
conv = nn.Conv2d(1, 1, kernel_size=2, bias=False)
conv.weight.data = K.view(1, 1, 2, 2)
X_batch = X.view(1, 1, 3, 3)  # format (batch, canal, H, W)
Y_torch = conv(X_batch).squeeze()
print(f'\nY PyTorch (identique) =\n{Y_torch.data.numpy()}')
print(f'Cohérence : {torch.allclose(Y_manual, Y_torch.detach(), atol=1e-5)}')

In [ ]:
# === Détection de contours (application des filtres) ===
X_edge = torch.ones((6, 8))
X_edge[:, 2:6] = 0  # bande noire au milieu
K_edge = torch.tensor([[1., -1.]])  # filtre de contour horizontal
Y_edge = corr2d(X_edge, K_edge)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(X_edge, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Image originale (blanc/noir)', fontweight='bold')
axes[1].imshow(K_edge, cmap='RdBu', vmin=-1, vmax=1)
axes[1].set_title('Filtre K = [1, -1]', fontweight='bold')
axes[2].imshow(Y_edge, cmap='RdBu', vmin=-1, vmax=1)
axes[2].set_title('Sortie : contours détectés', fontweight='bold')
plt.tight_layout()
plt.savefig('p2_contours.png', dpi=120)
plt.show()
print('Interprétation : valeurs +1 = passage clair→sombre, -1 = sombre→clair, 0 = pas de variation.')

In [ ]:
# === II.3 — Implémentation manuelle du pooling ===
def max_pool2d_manual(X, pool_size=(2, 2), stride=None):
    """Max-pooling 2D manuel."""
    ph, pw = pool_size
    sh, sw = stride if stride else pool_size
    Hout = (X.shape[0] - ph) // sh + 1
    Wout = (X.shape[1] - pw) // sw + 1
    Y = torch.zeros(Hout, Wout)
    for i in range(Hout):
        for j in range(Wout):
            region = X[i*sh:i*sh+ph, j*sw:j*sw+pw]
            Y[i, j] = region.max()  # maximum de la région
    return Y

def avg_pool2d_manual(X, pool_size=(2, 2), stride=None):
    """Average-pooling 2D manuel."""
    ph, pw = pool_size
    sh, sw = stride if stride else pool_size
    Hout = (X.shape[0] - ph) // sh + 1
    Wout = (X.shape[1] - pw) // sw + 1
    Y = torch.zeros(Hout, Wout)
    for i in range(Hout):
        for j in range(Wout):
            region = X[i*sh:i*sh+ph, j*sw:j*sw+pw]
            Y[i, j] = region.mean()  # moyenne de la région
    return Y

X_pool = torch.tensor([[1., 3., 2., 4.],
                        [5., 6., 7., 8.],
                        [3., 2., 1., 0.],
                        [1., 2., 3., 4.]])

print(f'Entrée (4×4) :\n{X_pool.numpy()}')
print(f'\nMax-pool (2×2) :\n{max_pool2d_manual(X_pool).numpy()}')
print(f'Avg-pool (2×2) :\n{avg_pool2d_manual(X_pool).numpy()}')

# Comparaison avec PyTorch
Xb = X_pool.view(1, 1, 4, 4)
mp_torch = F.max_pool2d(Xb, kernel_size=2).squeeze()
ap_torch = F.avg_pool2d(Xb, kernel_size=2).squeeze()
print(f'\nMax-pool PyTorch (identique) : {torch.allclose(max_pool2d_manual(X_pool), mp_torch.detach())}')
print(f'Avg-pool PyTorch (identique) : {torch.allclose(avg_pool2d_manual(X_pool), ap_torch.detach())}')
print('\nRôle du pooling : réduire la taille spatiale, augmenter le champ récepteur,')
print('réduire la sensibilité aux translations locales.')

In [ ]:
# === II.4 — Calculs de padding et stride ===
def output_size(H, W, k_h, k_w, p_h=0, p_w=0, s_h=1, s_w=1):
    """Formule de taille de sortie d'une convolution."""
    H_out = (H - k_h + 2*p_h) // s_h + 1
    W_out = (W - k_w + 2*p_w) // s_w + 1
    return H_out, W_out

configs = [
    (8, 8, 3, 3, 0, 0, 1, 1, 'Pas de padding, stride=1'),
    (8, 8, 3, 3, 1, 1, 1, 1, 'Padding=1 → conserve la taille'),
    (8, 8, 3, 3, 1, 1, 2, 2, 'Padding=1, stride=2 → division par 2'),
    (28, 28, 5, 5, 0, 0, 1, 1, 'MNIST 28x28, noyau 5x5'),
    (28, 28, 5, 5, 2, 2, 1, 1, 'MNIST 28x28, noyau 5x5, padding=2'),
]

print(f'{"Configuration":45s} | {"H_out":6s} | {"W_out":6s}')
print('-' * 65)
for H, W, kh, kw, ph, pw, sh, sw, desc in configs:
    Ho, Wo = output_size(H, W, kh, kw, ph, pw, sh, sw)
    print(f'{desc:45s} | {Ho:6d} | {Wo:6d}')

In [ ]:
# === II.5 — Chargement Fashion-MNIST ===
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # normalisation [-1, 1]
])

train_fmnist = torchvision.datasets.FashionMNIST(
    root='./data', train=True,  download=True, transform=transform)
test_fmnist  = torchvision.datasets.FashionMNIST(
    root='./data', train=False, download=True, transform=transform)

train_loader_cnn = DataLoader(train_fmnist, batch_size=128, shuffle=True,  num_workers=0)
test_loader_cnn  = DataLoader(test_fmnist,  batch_size=128, shuffle=False, num_workers=0)

class_names = ['T-shirt', 'Pantalon', 'Pull', 'Robe', 'Manteau',
               'Sandale', 'Chemise', 'Sneaker', 'Sac', 'Bottine']

print(f'Train : {len(train_fmnist)} images  |  Test : {len(test_fmnist)} images')
print(f'Taille des images : 28×28 pixels, 1 canal (niveaux de gris)')
print(f'Nombre de classes : 10')

# Visualisation
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for idx, ax in enumerate(axes.flat):
    img, label = train_fmnist[idx * 200]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(class_names[label], fontsize=9)
    ax.axis('off')
plt.suptitle('Fashion-MNIST — Exemples', fontweight='bold')
plt.tight_layout()
plt.savefig('p2_fmnist_examples.png', dpi=120)
plt.show()

In [ ]:
# === II.6 — MLP baseline sur Fashion-MNIST (point de référence) ===
class MLPBaseline(nn.Module):
    """MLP simple sur images aplaties."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, 512),  # 784 * 512 = 401 408 paramètres
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 10)
        )
    def forward(self, x):
        return self.net(x)

mlp_base = MLPBaseline().to(device)
print(f'MLP Baseline — Paramètres : {sum(p.numel() for p in mlp_base.parameters()):,}')

In [ ]:
# === II.7 — Architecture LeNet (inspirée de LeCun 1998) ===
class LeNet(nn.Module):
    """LeNet-5 adapté pour Fashion-MNIST (28x28, 10 classes)."""
    def __init__(self):
        super().__init__()
        # Bloc convolutionnel 1 : extrait les motifs locaux bas niveau
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 6, kernel_size=5, padding=2),  # 28x28 → 28x28 (6 filtres)
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)       # 28x28 → 14x14
        )
        # Bloc convolutionnel 2 : motifs plus complexes
        self.conv2 = nn.Sequential(
            nn.Conv2d(6, 16, kernel_size=5),            # 14x14 → 10x10 (16 filtres)
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)       # 10x10 → 5x5
        )
        # Classificateur FC
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16*5*5, 120),  # 16*5*5 = 400 features
            nn.ReLU(),
            nn.Linear(120, 84),
            nn.ReLU(),
            nn.Linear(84, 10)
        )

    def forward(self, x):
        x = self.conv2(self.conv1(x))
        return self.fc(x)

lenet = LeNet().to(device)
print('=== Architecture LeNet ===')
print(lenet)
params_lenet = sum(p.numel() for p in lenet.parameters())
params_mlp   = sum(p.numel() for p in mlp_base.parameters())
print(f'\nParamètres LeNet    : {params_lenet:,}')
print(f'Paramètres MLP Base : {params_mlp:,}')
print(f'Réduction           : {params_mlp/params_lenet:.1f}x moins de paramètres pour LeNet')

# Inspection des dimensions
x_test_img = torch.zeros(1, 1, 28, 28).to(device)
print('\n=== Dimensions à travers le réseau ===')
out = x_test_img
for name, layer in [('Input', None), ('Conv1', lenet.conv1), ('Conv2', lenet.conv2)]:
    if layer is not None:
        out = layer(out)
    print(f'  {name:8s} : {list(out.shape)}')

In [ ]:
# === Fonction d'entraînement générique pour CNN ===
def train_cnn(model, train_loader, test_loader, epochs=10, lr=1e-3, model_name=''):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    history = {'train_acc': [], 'test_acc': [], 'train_loss': []}

    for epoch in range(epochs):
        model.train()
        correct, total, total_loss = 0, 0, 0.0
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            out  = model(X)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            correct    += (out.argmax(1) == y).sum().item()
            total      += y.size(0)
            total_loss += loss.item() * y.size(0)

        train_acc  = correct / total
        train_loss = total_loss / total

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for X, y in test_loader:
                X, y = X.to(device), y.to(device)
                correct += (model(X).argmax(1) == y).sum().item()
                total   += y.size(0)
        test_acc = correct / total

        history['train_acc'].append(train_acc)
        history['train_loss'].append(train_loss)
        history['test_acc'].append(test_acc)

        if (epoch + 1) % 2 == 0 or epoch == 0:
            print(f'[{model_name}] Epoch {epoch+1:2d}/{epochs} | '
                  f'Train: {train_acc:.4f} | Test: {test_acc:.4f} | Loss: {train_loss:.4f}')

    return history

print('Fonction train_cnn() définie.')

In [ ]:
# === Entraînement MLP Baseline vs LeNet ===
print('=== Entraînement MLP Baseline ===')
hist_mlp = train_cnn(mlp_base, train_loader_cnn, test_loader_cnn, epochs=10, model_name='MLP')

print('\n=== Entraînement LeNet ===')
hist_lenet = train_cnn(lenet, train_loader_cnn, test_loader_cnn, epochs=10, model_name='LeNet')

# Comparaison visuelle
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(hist_mlp['train_acc'],   label='MLP Train',  color='coral')
axes[0].plot(hist_mlp['test_acc'],    label='MLP Test',   color='coral',     linestyle='--')
axes[0].plot(hist_lenet['train_acc'], label='LeNet Train', color='steelblue')
axes[0].plot(hist_lenet['test_acc'],  label='LeNet Test',  color='steelblue', linestyle='--')
axes[0].set_title('Accuracy : MLP vs LeNet', fontweight='bold')
axes[0].set_xlabel('Époque')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

final_data = {'MLP': hist_mlp['test_acc'][-1], 'LeNet': hist_lenet['test_acc'][-1]}
axes[1].bar(final_data.keys(), final_data.values(), color=['coral', 'steelblue'])
for k, v in final_data.items():
    axes[1].text(list(final_data.keys()).index(k), v + 0.002, f'{v:.4f}',
                 ha='center', fontweight='bold')
axes[1].set_title('Accuracy finale sur le test set', fontweight='bold')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0.80, 0.95)

plt.tight_layout()
plt.savefig('p2_mlp_vs_lenet.png', dpi=120)
plt.show()
print(f'\nMLP Baseline : {hist_mlp["test_acc"][-1]:.4f}')
print(f'LeNet        : {hist_lenet["test_acc"][-1]:.4f}')

In [ ]:
# === II.8 — Étude expérimentale des hyperparamètres CNN ===

def build_cnn(padding=2, stride=1, pool='max', n_filters=6, use_conv1x1=False):
    """CNN paramétrable pour l'étude expérimentale."""
    pool_layer = nn.MaxPool2d(2, 2) if pool == 'max' else nn.AvgPool2d(2, 2)
    layers = [
        nn.Conv2d(1, n_filters, kernel_size=5, padding=padding, stride=stride),
        nn.ReLU(), pool_layer
    ]
    if use_conv1x1:
        layers += [nn.Conv2d(n_filters, n_filters*2, kernel_size=1), nn.ReLU()]
        out_c = n_filters * 2
    else:
        out_c = n_filters

    # Calcul de la taille de sortie après pooling
    H_after_conv  = (28 - 5 + 2*padding) // stride + 1
    H_after_pool  = H_after_conv // 2
    flat_size = out_c * H_after_pool * H_after_pool

    layers += [nn.Flatten(), nn.Linear(flat_size, 128), nn.ReLU(), nn.Linear(128, 10)]
    return nn.Sequential(*layers).to(device)

# Expériences
experiments = [
    {'padding':2, 'stride':1, 'pool':'max', 'n_filters':6,  'use_conv1x1':False, 'nom':'Baseline'},
    {'padding':0, 'stride':1, 'pool':'max', 'n_filters':6,  'use_conv1x1':False, 'nom':'Sans padding'},
    {'padding':2, 'stride':2, 'pool':'max', 'n_filters':6,  'use_conv1x1':False, 'nom':'Stride=2'},
    {'padding':2, 'stride':1, 'pool':'avg', 'n_filters':6,  'use_conv1x1':False, 'nom':'Avg pooling'},
    {'padding':2, 'stride':1, 'pool':'max', 'n_filters':16, 'use_conv1x1':False, 'nom':'16 filtres'},
    {'padding':2, 'stride':1, 'pool':'max', 'n_filters':6,  'use_conv1x1':True,  'nom':'Conv 1x1'},
]

exp_results = []
for config in experiments:
    nom = config.pop('nom')
    model_exp = build_cnn(**config)
    params = sum(p.numel() for p in model_exp.parameters())
    hist = train_cnn(model_exp, train_loader_cnn, test_loader_cnn, epochs=5, model_name=nom)
    acc = hist['test_acc'][-1]
    exp_results.append({'Configuration': nom, 'Params': params,
                        'Test Acc (5ep)': round(acc, 4)})
    config['nom'] = nom
    print(f'  {nom:20s} | params={params:6d} | acc={acc:.4f}')

df_exp = pd.DataFrame(exp_results)
print('\n=== TABLEAU EXPÉRIMENTAL CNN ===')
display(df_exp)

In [ ]:
# === II.9 — Visualisation des cartes de caractéristiques ===
lenet.eval()
sample_img, sample_label = test_fmnist[0]
sample_tensor = sample_img.unsqueeze(0).to(device)

# Extraction des feature maps de la couche conv1
with torch.no_grad():
    feat_maps_c1 = lenet.conv1[0](sample_tensor)  # après conv1 seulement
    feat_maps_c1 = F.relu(feat_maps_c1)

fig, axes = plt.subplots(1, 7, figsize=(16, 3))
axes[0].imshow(sample_img.squeeze(), cmap='gray')
axes[0].set_title(f'Input\n{class_names[sample_label]}', fontweight='bold')
axes[0].axis('off')

for i in range(6):
    axes[i+1].imshow(feat_maps_c1[0, i].cpu(), cmap='viridis')
    axes[i+1].set_title(f'Filtre {i+1}', fontsize=9)
    axes[i+1].axis('off')

plt.suptitle('Cartes de caractéristiques — Conv1 de LeNet', fontweight='bold')
plt.tight_layout()
plt.savefig('p2_feature_maps.png', dpi=120)
plt.show()
print('Observation : chaque filtre détecte un motif différent (contours, textures).')
print('Les filtres appris automatiquement détectent des features pertinentes pour Fashion-MNIST.')

## II.10 — Question de synthèse — Partie II

**Pourquoi un CNN est-il plus pertinent qu'un MLP pour la classification d'images, et comment les choix de padding, stride, pooling et profondeur influencent-ils les performances ?**

### Réponse

**Supériorité du CNN sur le MLP pour les images :**
- MLP avec 784 entrées × 512 neurones = 401 408 paramètres pour une seule couche → explosion paramétrique
- LeNet (6+16 filtres) = environ **60 000 paramètres** → 6× moins avec de meilleures performances
- Le CNN exploite la **localité spatiale** : deux pixels voisins sont liés
- Le **partage des poids** (même filtre partout) réduit le nombre de paramètres et apporte l'invariance par translation
- La **hiérarchie** des représentations (bords → formes → objets) est naturellement apprise

**Impact des hyperparamètres (validé expérimentalement) :**
- **Padding=2** : conserve la taille spatiale → plus d'information aux bords → meilleure accuracy
- **Stride=2** : réduit la taille de sortie (comme le pooling) mais peut perdre de l'information
- **Max vs Avg pooling** : max-pooling détecte mieux les motifs saillants ; avg-pooling lisse davantage
- **Plus de filtres (16 vs 6)** : capture plus de types de motifs → meilleure accuracy au prix de plus de paramètres
- **Convolution 1×1** : projette les canaux sans modifier la dimension spatiale → compression d'information

**Conclusion :** Le CNN est le bon inductive bias pour les images. Ses hypothèses structurelles
(localité, partage, hiérarchie) correspondent à la nature des données visuelles.


---
# PARTIE III — RNN, LSTM, GRU et Seq2Seq
## Modélisation de séquences et traduction automatique
---

## III.1 — Fondements théoriques des modèles séquentiels

### Modèle de langage et règle de chaîne
Un modèle de langage attribue une probabilité à une séquence :
$$P(x_1, x_2, \ldots, x_T) = \prod_{t=1}^{T} P(x_t \mid x_1, \ldots, x_{t-1})$$

L'idée fondatrice : **prédire le prochain token à partir du passé**.

### Perplexité
Mesure la qualité d'un modèle de langage :
$$\text{PPL} = \exp\left(-\frac{1}{T}\sum_{t=1}^{T}\log P(x_t \mid x_{<t})\right)$$
- PPL faible → modèle confiant et correct
- PPL élevée → modèle incertain
- PPL = 1 → parfait ; PPL = |V| → modèle aléatoire uniforme

### RNN — Équation de récurrence
$$h_t = \phi(W_{xh}x_t + W_{hh}h_{t-1} + b_h)$$
$$o_t = W_{hq}h_t + b_q, \quad \hat{y}_t = \text{softmax}(o_t)$$

### BPTT (Backpropagation Through Time)
Le gradient doit traverser tous les pas temporels :
$$\frac{\partial\mathcal{L}}{\partial h_k} = \sum_{t\geq k}\frac{\partial\mathcal{L}}{\partial h_t}\prod_{i=k+1}^{t}\frac{\partial h_i}{\partial h_{i-1}}$$
→ **Gradient évanescent** ou **explosif** pour les longues séquences.

### Gradient Clipping
Si $\|g\| > \theta$ : $g \leftarrow \min\left(1, \frac{\theta}{\|g\|}\right) g$


In [ ]:
# === III.2 — Préparation des données : corpus anglais-français ===
# Corpus de 80 paires anglais → français
raw_pairs = [
    ('go .', 'va .'), ('hi .', 'salut !'), ('run !', 'cours !'),
    ('i am ok', 'je vais bien'), ('he is here', 'il est ici'),
    ('she is nice', 'elle est gentille'), ('we are home', 'nous sommes a la maison'),
    ('they eat well', 'ils mangent bien'), ('i like cats', 'j aime les chats'),
    ('she reads books', 'elle lit des livres'), ('he plays piano', 'il joue du piano'),
    ('we study french', 'nous etudions le francais'),
    ('it is cold', 'il fait froid'), ('it is hot', 'il fait chaud'),
    ('the cat is black', 'le chat est noir'), ('the dog runs', 'le chien court'),
    ('i am tired', 'je suis fatigue'), ('she is happy', 'elle est heureuse'),
    ('he is sad', 'il est triste'), ('i want water', 'je veux de l eau'),
    ('open the door', 'ouvrez la porte'), ('close the window', 'fermez la fenetre'),
    ('i love you', 'je t aime'), ('good morning', 'bonjour'),
    ('good night', 'bonne nuit'), ('thank you', 'merci'),
    ('you are welcome', 'de rien'), ('see you later', 'a bientot'),
    ('i am a student', 'je suis etudiant'), ('she is a teacher', 'elle est professeur'),
    ('i am hungry', 'j ai faim'), ('i am thirsty', 'j ai soif'),
    ('the weather is nice', 'le temps est beau'), ('it rains today', 'il pleut aujourd hui'),
    ('i read a book', 'je lis un livre'), ('he writes a letter', 'il ecrit une lettre'),
    ('they play football', 'ils jouent au football'),
    ('she sings well', 'elle chante bien'), ('we dance together', 'nous dansons ensemble'),
    ('i go to school', 'je vais a l ecole'), ('he goes to work', 'il va au travail'),
    ('she comes home', 'elle rentre a la maison'),
    ('the sun shines', 'le soleil brille'), ('the moon is full', 'la lune est pleine'),
    ('i like music', 'j aime la musique'), ('she plays violin', 'elle joue du violon'),
    ('he paints well', 'il peint bien'), ('we travel often', 'nous voyageons souvent'),
    ('i speak french', 'je parle francais'), ('she speaks english', 'elle parle anglais'),
]

# === Tokenisation et vocabulaire ===
SOS, EOS, PAD, UNK = '<sos>', '<eos>', '<pad>', '<unk>'

class Vocabulaire:
    """Vocabulaire avec tokens spéciaux."""
    def __init__(self):
        self.mot2idx = {PAD: 0, SOS: 1, EOS: 2, UNK: 3}
        self.idx2mot = {v: k for k, v in self.mot2idx.items()}

    def ajouter(self, texte):
        for mot in texte.split():
            if mot not in self.mot2idx:
                idx = len(self.mot2idx)
                self.mot2idx[mot] = idx
                self.idx2mot[idx] = mot

    def encoder(self, texte, max_len=12):
        tokens = [self.mot2idx.get(m, 3) for m in texte.split()]
        tokens += [2]  # EOS
        tokens += [0] * (max_len - len(tokens))  # PAD
        return tokens[:max_len]

    def decoder(self, indices):
        return ' '.join([self.idx2mot.get(i, UNK) for i in indices
                         if i not in [0, 1, 2]])

    def __len__(self): return len(self.mot2idx)

vocab_en = Vocabulaire()
vocab_fr = Vocabulaire()
for en, fr in raw_pairs:
    vocab_en.ajouter(en)
    vocab_fr.ajouter(fr)

MAX_LEN = 12
src_data = torch.tensor([vocab_en.encoder(en, MAX_LEN) for en, _ in raw_pairs])
tgt_data = torch.tensor([vocab_fr.encoder(fr, MAX_LEN) for _, fr in raw_pairs])

print(f'Vocabulaire anglais : {len(vocab_en)} tokens')
print(f'Vocabulaire français : {len(vocab_fr)} tokens')
print(f'Données : {src_data.shape}  (paires × longueur max)')
print(f'\nExemple encodé EN : {src_data[0].tolist()}')
print(f'Exemple encodé FR : {tgt_data[0].tolist()}')
print(f'Décodé EN : {vocab_en.decoder(src_data[0].tolist())}')
print(f'Décodé FR : {vocab_fr.decoder(tgt_data[0].tolist())}')

In [ ]:
# === III.3 — Implémentation d'un RNN simple pour modèle de langage ===

# -- Modèle de langage sur séquences françaises --
class RNNLanguageModel(nn.Module):
    """RNN simple : h_t = tanh(W_xh * x_t + W_hh * h_{t-1} + b_h)"""
    def __init__(self, vocab_size, embed_dim=32, hidden_size=64, num_layers=1):
        super().__init__()
        self.embedding   = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn         = nn.RNN(embed_dim, hidden_size, num_layers,
                                  batch_first=True, nonlinearity='tanh')
        self.fc          = nn.Linear(hidden_size, vocab_size)
        self.hidden_size = hidden_size
        self.num_layers  = num_layers

    def forward(self, x, h=None):
        emb = self.embedding(x)          # (batch, seq, embed_dim)
        out, h_n = self.rnn(emb, h)      # out: (batch, seq, hidden)
        logits = self.fc(out)            # (batch, seq, vocab_size)
        return logits, h_n

    def init_hidden(self, batch_size):
        return torch.zeros(self.num_layers, batch_size, self.hidden_size).to(device)

rnn_lm = RNNLanguageModel(len(vocab_fr)).to(device)
print('RNN Language Model créé.')
print(f'Paramètres : {sum(p.numel() for p in rnn_lm.parameters()):,}')

In [ ]:
# === III.4 — LSTM (Long Short-Term Memory) ===
# Portes LSTM :
# f_t = sigma(W_f*x_t + U_f*h_{t-1} + b_f)   # porte d'oubli
# i_t = sigma(W_i*x_t + U_i*h_{t-1} + b_i)   # porte d'entrée
# c~_t = tanh(W_c*x_t + U_c*h_{t-1} + b_c)   # candidat
# c_t = f_t ⊙ c_{t-1} + i_t ⊙ c~_t          # état de cellule
# o_t = sigma(W_o*x_t + U_o*h_{t-1} + b_o)   # porte de sortie
# h_t = o_t ⊙ tanh(c_t)

class LSTMLanguageModel(nn.Module):
    """LSTM : résout les problèmes de gradient évanescent via l'état de cellule c_t."""
    def __init__(self, vocab_size, embed_dim=32, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm      = nn.LSTM(embed_dim, hidden_size, num_layers,
                                 batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.fc        = nn.Linear(hidden_size, vocab_size)
        self.hidden_size = hidden_size
        self.num_layers  = num_layers

    def forward(self, x, hidden=None):
        emb = self.embedding(x)
        out, (h_n, c_n) = self.lstm(emb, hidden)
        return self.fc(out), (h_n, c_n)

    def init_hidden(self, batch):
        h = torch.zeros(self.num_layers, batch, self.hidden_size).to(device)
        c = torch.zeros(self.num_layers, batch, self.hidden_size).to(device)
        return (h, c)

lstm_lm = LSTMLanguageModel(len(vocab_fr)).to(device)
print('LSTM Language Model créé.')
print(f'Paramètres : {sum(p.numel() for p in lstm_lm.parameters()):,}')
print('\nAvantage clé du LSTM : l\'état de cellule c_t offre un "autoroute" pour l\'information')
print('qui court-circuite les problèmes de gradient évanescent.')

In [ ]:
# === III.5 — GRU (Gated Recurrent Unit) ===
# GRU simplifie LSTM avec 2 portes au lieu de 3 :
# z_t = sigma(W_z*x_t + U_z*h_{t-1})   # porte de mise à jour
# r_t = sigma(W_r*x_t + U_r*h_{t-1})   # porte de reset
# h~_t = tanh(W_h*x_t + U_h*(r_t ⊙ h_{t-1}))
# h_t = (1 - z_t) ⊙ h_{t-1} + z_t ⊙ h~_t

class GRULanguageModel(nn.Module):
    """GRU : alternative plus légère au LSTM, souvent performances similaires."""
    def __init__(self, vocab_size, embed_dim=32, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru       = nn.GRU(embed_dim, hidden_size, num_layers,
                                batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.fc        = nn.Linear(hidden_size, vocab_size)
        self.hidden_size = hidden_size
        self.num_layers  = num_layers

    def forward(self, x, h=None):
        emb = self.embedding(x)
        out, h_n = self.gru(emb, h)
        return self.fc(out), h_n

    def init_hidden(self, batch):
        return torch.zeros(self.num_layers, batch, self.hidden_size).to(device)

gru_lm = GRULanguageModel(len(vocab_fr)).to(device)
print('GRU Language Model créé.')
print(f'Paramètres : {sum(p.numel() for p in gru_lm.parameters()):,}')
print('\nComparaison LSTM vs GRU :')
print('  LSTM : 3 portes (oubli, entrée, sortie) + état de cellule séparé')
print('  GRU  : 2 portes (mise à jour, reset) — plus simple, souvent aussi efficace')

In [ ]:
# === III.6 — BPTT et Gradient Clipping ===

def train_lm(model, data, vocab_size, epochs=30, lr=1e-3, clip=1.0, model_name=''):
    """Entraînement d'un modèle de langage avec BPTT et gradient clipping."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(ignore_index=0)  # ignore le padding
    history = {'loss': [], 'ppl': [], 'grad_norm': []}

    data = data.to(device)

    for epoch in range(epochs):
        model.train()
        # Entrée : tokens 0..T-2  |  Cible : tokens 1..T-1 (prédiction du suivant)
        src = data[:, :-1]
        tgt = data[:, 1:]

        hidden = model.init_hidden(data.shape[0])
        if isinstance(hidden, tuple):  # LSTM retourne (h, c)
            hidden = tuple(h.detach() for h in hidden)
        else:
            hidden = hidden.detach()

        optimizer.zero_grad()
        logits, _ = model(src, hidden)          # (batch, seq, vocab)
        logits = logits.reshape(-1, vocab_size) # (batch*seq, vocab)
        tgt_flat = tgt.reshape(-1)              # (batch*seq,)
        loss = criterion(logits, tgt_flat)
        loss.backward()                         # BPTT : gradient à travers le temps

        # === Gradient Clipping ===
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip)
        optimizer.step()

        ppl = math.exp(min(loss.item(), 50))
        history['loss'].append(loss.item())
        history['ppl'].append(ppl)
        history['grad_norm'].append(grad_norm.item())

        if (epoch + 1) % 10 == 0:
            print(f'[{model_name}] Epoch {epoch+1:3d} | Loss={loss.item():.4f} | '
                  f'PPL={ppl:.2f} | GradNorm={grad_norm:.4f}')

    return history

print('=== Entraînement des 3 modèles de langage ===')
print('--- RNN ---')
hist_rnn = train_lm(rnn_lm, tgt_data, len(vocab_fr), epochs=30, model_name='RNN')
print('--- LSTM ---')
hist_lstm = train_lm(lstm_lm, tgt_data, len(vocab_fr), epochs=30, model_name='LSTM')
print('--- GRU ---')
hist_gru = train_lm(gru_lm, tgt_data, len(vocab_fr), epochs=30, model_name='GRU')

In [ ]:
# === Comparaison RNN / LSTM / GRU ===
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for name, hist, col in [('RNN', hist_rnn, 'coral'), ('LSTM', hist_lstm, 'steelblue'),
                         ('GRU', hist_gru, 'green')]:
    axes[0].plot(hist['loss'],      label=name, color=col)
    axes[1].plot(hist['ppl'],       label=name, color=col)
    axes[2].plot(hist['grad_norm'], label=name, color=col)

axes[0].set_title('Loss (CrossEntropy)',   fontweight='bold')
axes[1].set_title('Perplexité',            fontweight='bold')
axes[2].set_title('Norme du gradient',     fontweight='bold')
for ax in axes:
    ax.set_xlabel('Époque')
    ax.legend()

plt.suptitle('Comparaison RNN / LSTM / GRU — Modèle de langage', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('p3_rnn_comparison.png', dpi=120)
plt.show()

print('=== TABLEAU COMPARATIF FINAL ===')
for name, hist in [('RNN', hist_rnn), ('LSTM', hist_lstm), ('GRU', hist_gru)]:
    print(f'  {name:6s} | Loss finale: {hist["loss"][-1]:.4f} | '
          f'PPL finale: {hist["ppl"][-1]:.2f}')
print('\nObservation : LSTM et GRU convergent plus vite et atteignent une PPL plus basse.')
print('Le RNN simple souffre du gradient évanescent sur les séquences longues.')
print('GRU ≈ LSTM en performance avec moins de paramètres.')

In [ ]:
# === III.7 — Système Seq2Seq avec encodeur et décodeur récurrents ===

class Encodeur(nn.Module):
    """Encode la séquence source en un vecteur de contexte (état caché final)."""
    def __init__(self, vocab_size, embed_dim=32, hidden_size=64, num_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(embed_dim, hidden_size, num_layers, batch_first=True)

    def forward(self, x):
        emb = self.embedding(x)         # (batch, seq, embed)
        _, h_n = self.gru(emb)          # h_n = état caché final = vecteur de contexte
        return h_n

class Decodeur(nn.Module):
    """Décode token par token à partir du vecteur de contexte de l'encodeur."""
    def __init__(self, vocab_size, embed_dim=32, hidden_size=64, num_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(embed_dim, hidden_size, num_layers, batch_first=True)
        self.fc  = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, h):
        # x : token courant (batch, 1)
        # h : état caché du pas précédent
        emb = self.embedding(x)         # (batch, 1, embed)
        out, h_n = self.gru(emb, h)     # (batch, 1, hidden)
        logits = self.fc(out.squeeze(1))# (batch, vocab_size)
        return logits, h_n

class Seq2Seq(nn.Module):
    """Architecture encodeur-décodeur complète."""
    def __init__(self, src_vocab, tgt_vocab, embed_dim=32, hidden_size=64):
        super().__init__()
        self.encodeur = Encodeur(src_vocab, embed_dim, hidden_size)
        self.decodeur = Decodeur(tgt_vocab, embed_dim, hidden_size)

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        """
        Teacher Forcing : avec probabilité p, on donne le vrai token précédent
        au décodeur plutôt que sa propre prédiction → stabilise l'entraînement.
        """
        batch_size = src.shape[0]
        tgt_len    = tgt.shape[1]
        tgt_vocab  = self.decodeur.fc.out_features

        # Encodage
        context = self.encodeur(src)              # vecteur de contexte
        h = context

        # Token initial : SOS
        x = torch.full((batch_size, 1), 1, dtype=torch.long).to(device)  # SOS=1
        all_logits = []

        for t in range(tgt_len - 1):
            logits, h = self.decodeur(x, h)
            all_logits.append(logits.unsqueeze(1))  # (batch, 1, vocab)

            # Teacher Forcing
            if random.random() < teacher_forcing_ratio:
                x = tgt[:, t+1:t+2]  # vrai token suivant
            else:
                x = logits.argmax(dim=-1).unsqueeze(1)  # token prédit

        return torch.cat(all_logits, dim=1)  # (batch, tgt_len-1, vocab)

seq2seq = Seq2Seq(len(vocab_en), len(vocab_fr)).to(device)
print('=== Seq2Seq Encodeur-Décodeur ===')
print(seq2seq)
print(f'\nParamètres totaux : {sum(p.numel() for p in seq2seq.parameters()):,}')

In [ ]:
# === Entraînement du Seq2Seq ===
def train_seq2seq(model, src_data, tgt_data, epochs=50, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(ignore_index=0)
    history = {'loss': []}

    src = src_data.to(device)
    tgt = tgt_data.to(device)

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        # teacher_forcing décroit progressivement
        tf_ratio = max(0.1, 0.9 - epoch * 0.01)
        output = model(src, tgt, teacher_forcing_ratio=tf_ratio)

        # output : (batch, tgt_len-1, vocab_fr)
        output_flat = output.reshape(-1, len(vocab_fr))
        tgt_flat    = tgt[:, 1:].reshape(-1)  # décalage d'un token (on prédit le suivant)

        loss = criterion(output_flat, tgt_flat)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # gradient clipping
        optimizer.step()

        history['loss'].append(loss.item())
        if (epoch + 1) % 10 == 0:
            print(f'Epoch {epoch+1:3d}/{epochs} | Loss: {loss.item():.4f} | '
                  f'Teacher Forcing: {tf_ratio:.2f}')

    return history

print('=== Entraînement Seq2Seq ===')
hist_s2s = train_seq2seq(seq2seq, src_data, tgt_data, epochs=80)

plt.figure(figsize=(10, 4))
plt.plot(hist_s2s['loss'], color='purple', linewidth=1.5)
plt.title('Courbe de perte — Seq2Seq (Teacher Forcing)', fontweight='bold')
plt.xlabel('Époque')
plt.ylabel('CrossEntropy Loss')
plt.tight_layout()
plt.savefig('p3_seq2seq_loss.png', dpi=120)
plt.show()

In [ ]:
# === III.8 — Décodage glouton (Greedy) ===
def decode_greedy(model, src_sentence, vocab_en, vocab_fr, max_len=12):
    """Décodage glouton : à chaque étape, on choisit le token de probabilité maximale."""
    model.eval()
    with torch.no_grad():
        src = torch.tensor([vocab_en.encoder(src_sentence, max_len)]).to(device)
        context = model.encodeur(src)
        h = context
        x = torch.tensor([[1]]).to(device)  # SOS token
        result = []
        for _ in range(max_len):
            logits, h = model.decodeur(x, h)
            token = logits.argmax(dim=-1).item()
            if token == 2:  # EOS
                break
            if token not in [0, 1, 2, 3]:
                result.append(vocab_fr.idx2mot.get(token, UNK))
            x = torch.tensor([[token]]).to(device)
    return ' '.join(result)

print('=== DÉCODAGE GLOUTON ===')
test_phrases = [
    ('i am ok',       'je vais bien'),
    ('she is nice',   'elle est gentille'),
    ('good morning',  'bonjour'),
    ('i love you',    'je t aime'),
    ('i am hungry',   'j ai faim'),
]
for src_phrase, tgt_ref in test_phrases:
    pred = decode_greedy(seq2seq, src_phrase, vocab_en, vocab_fr)
    print(f'  EN : {src_phrase:25s}  →  FR prédit : {pred:30s}  (réf: {tgt_ref})')

In [ ]:
# === III.9 — Beam Search (largeur k=3) ===
def decode_beam(model, src_sentence, vocab_en, vocab_fr, beam_width=3, max_len=12):
    """
    Beam Search : maintient les k meilleures hypothèses à chaque étape.
    Évite les erreurs de décisions prématurées du décodage glouton.
    Score = log P(y_1,...,y_t) pour éviter les problèmes numériques.
    """
    model.eval()
    with torch.no_grad():
        src = torch.tensor([vocab_en.encoder(src_sentence, max_len)]).to(device)
        h0  = model.encodeur(src)  # (1, 1, hidden)

        # Chaque hypothèse : (score_log, tokens_générés, état_caché)
        beams = [(0.0, [1], h0)]   # SOS = token 1
        completed = []

        for _ in range(max_len):
            new_beams = []
            for score, tokens, h in beams:
                x = torch.tensor([[tokens[-1]]]).to(device)
                logits, h_new = model.decodeur(x, h)
                log_probs = F.log_softmax(logits, dim=-1).squeeze(0)

                # Top-k tokens candidats
                topk_scores, topk_tokens = log_probs.topk(beam_width)
                for s, t in zip(topk_scores.tolist(), topk_tokens.tolist()):
                    new_score = score + s
                    new_tokens = tokens + [t]
                    if t == 2:  # EOS
                        # Pénalisation de longueur (évite les séquences trop courtes)
                        length_penalty = len(new_tokens) ** 0.6
                        completed.append((new_score / length_penalty, new_tokens))
                    else:
                        new_beams.append((new_score, new_tokens, h_new))

            if not new_beams:
                break
            beams = sorted(new_beams, key=lambda x: x[0], reverse=True)[:beam_width]

        if not completed:
            best_tokens = beams[0][1]
        else:
            best_tokens = sorted(completed, reverse=True)[0][1]

        return ' '.join([vocab_fr.idx2mot.get(t, UNK)
                         for t in best_tokens if t not in [0, 1, 2, 3]])

print('=== DÉCODAGE BEAM SEARCH (k=3) ===')
print(f'{"Source":30s} | {"Glouton":30s} | {"Beam Search":30s} | {"Référence":30s}')
print('-' * 125)
for src_phrase, tgt_ref in test_phrases:
    greedy_out = decode_greedy(seq2seq, src_phrase, vocab_en, vocab_fr)
    beam_out   = decode_beam(seq2seq, src_phrase, vocab_en, vocab_fr, beam_width=3)
    print(f'{src_phrase:30s} | {greedy_out:30s} | {beam_out:30s} | {tgt_ref:30s}')

In [ ]:
# === III.10 — Score BLEU ===
def bleu_score(candidate, reference, max_n=4):
    """
    BLEU = BP * exp(sum_n w_n * log p_n)
    p_n = précision des n-grammes de la candidate dans la référence
    BP  = Brevity Penalty (pénalise les traductions trop courtes)
    """
    cand_tokens = candidate.split()
    ref_tokens  = reference.split()

    if len(cand_tokens) == 0:
        return 0.0

    # Brevity Penalty
    BP = 1.0 if len(cand_tokens) >= len(ref_tokens) else \
         math.exp(1 - len(ref_tokens) / len(cand_tokens))

    log_sum = 0.0
    for n in range(1, max_n + 1):
        if len(cand_tokens) < n:
            break
        cand_ngrams = Counter([tuple(cand_tokens[i:i+n])
                               for i in range(len(cand_tokens)-n+1)])
        ref_ngrams  = Counter([tuple(ref_tokens[i:i+n])
                               for i in range(len(ref_tokens)-n+1)])
        matches = sum((cand_ngrams & ref_ngrams).values())
        total   = sum(cand_ngrams.values())
        if total == 0 or matches == 0:
            return 0.0
        log_sum += (1.0/max_n) * math.log(matches / total)

    return BP * math.exp(log_sum)

print('=== ÉVALUATION BLEU ===')
print(f'{"Source":25s} | {"BLEU Glouton":14s} | {"BLEU Beam":14s}')
print('-' * 60)
bleu_greedy_list, bleu_beam_list = [], []
for src_phrase, tgt_ref in test_phrases:
    greedy_out  = decode_greedy(seq2seq, src_phrase, vocab_en, vocab_fr)
    beam_out    = decode_beam(seq2seq, src_phrase, vocab_en, vocab_fr)
    bg = bleu_score(greedy_out, tgt_ref)
    bb = bleu_score(beam_out, tgt_ref)
    bleu_greedy_list.append(bg)
    bleu_beam_list.append(bb)
    print(f'{src_phrase:25s} | {bg:.4f}         | {bb:.4f}')

print(f'\nBLEU moyen Glouton : {np.mean(bleu_greedy_list):.4f}')
print(f'BLEU moyen Beam    : {np.mean(bleu_beam_list):.4f}')
print('\nObservation : le Beam Search explore davantage de chemins de décodage,')
print('ce qui améliore généralement le BLEU par rapport au décodage glouton.')

## III.11 — Question de synthèse — Partie III

**Dans quelle mesure les architectures récurrentes modélisent-elles efficacement une séquence réelle, et comment justifier le passage RNN → LSTM/GRU → Seq2Seq ?**

### Réponse

**1. Du RNN simple aux LSTM/GRU :**
Le RNN simple souffre du **gradient évanescent** : lors du BPTT, le produit de jacobiens
`∂h_i/∂h_{i-1}` tend vers 0 pour les longues séquences → le RNN "oublie" les dépendances lointaines.

Le LSTM résout ce problème via :
- L'**état de cellule c_t** : une mémoire à long terme que les portes contrôlent
- La **porte d'oubli** f_t décide quoi effacer
- La **porte d'entrée** i_t décide quoi écrire
- Cela crée un **chemin de gradient plus direct** → résout le gradient évanescent

Le GRU simplifie avec 2 portes (mise à jour + reset), souvent aussi efficace avec moins de paramètres.

**2. Du LSTM au Seq2Seq :**
Pour des tâches où l'entrée et la sortie ont des longueurs différentes (traduction), un LSTM
simple ne suffit pas. L'architecture **encodeur-décodeur** :
- L'**encodeur** compresse la séquence source en un **vecteur de contexte** h_N
- Le **décodeur** génère la séquence cible token par token, conditionné sur h_N
- Le **Teacher Forcing** accélère l'entraînement en donnant le vrai token précédent pendant l'entraînement

**Limites observées :**
- Le vecteur de contexte fixe est un **goulot d'étranglement** pour les longues séquences
- Solution → **mécanisme d'attention** (Transformer) qui permet au décodeur de regarder toute la séquence source
- Notre corpus est petit (50 paires) → les scores BLEU restent modestes mais le principe est validé


---
# Question transversale finale
---

**Comment le deep learning adapte-t-il ses architectures à la structure des données — tabulaire, image et séquentielle — et pourquoi un même paradigme d'apprentissage supervisé doit-il être décliné différemment selon la géométrie, la dépendance locale, la temporalité et la représentation des données ?**

## Réponse synthétique

Le deep learning repose sur un paradigme unique : **minimiser une perte empirique par descente de gradient**.
Pourtant, l'architecture optimale varie radicalement selon la nature des données :

| Critère | Données tabulaires → MLP | Images → CNN | Séquences → RNN/LSTM |
|---------|--------------------------|--------------|----------------------|
| **Géométrie** | Vecteur plat, features indépendantes | Grille 2D, pixels corrélés localement | Suite ordonnée, tokens interdépendants |
| **Inductive bias** | Toutes les features contribuent indépendamment | Localité + partage des poids | Mémoire récurrente (h_t) |
| **Invariance** | Permutation des features | Translation spatiale | Causalité temporelle |
| **Paramètres** | Dense : O(d² par couche) | Partagés : O(k²×C) par filtre | Partagés : O(h²) par pas de temps |
| **Limite principale** | Pas de structure spatiale/temporelle | Long range spatial difficile | Gradient évanescent, longueur fixe |

**Le lien conceptuel entre les 3 architectures :**

Toutes utilisent :
1. Des **transformations linéaires** (couches denses, convolutions, matrices récurrentes)
2. Des **non-linéarités** (ReLU, tanh, sigmoid)
3. La **rétropropagation** pour calculer les gradients
4. Un **optimiseur** (Adam) pour mettre à jour les paramètres

La différence réside dans **comment les paramètres sont partagés** :
- **MLP** : aucun partage → chaque connexion est indépendante
- **CNN** : partage spatial → même filtre à chaque position de l'image
- **RNN** : partage temporel → mêmes matrices W_xh et W_hh à chaque pas de temps

**Conclusion :** Choisir la bonne architecture, c'est encoder les **bonnes hypothèses structurelles** sur les données. 
Un MLP sur des images détruirait la géométrie 2D. Un CNN sur des séquences textuelles ignorerait l'ordre causal.
Un RNN sur des données tabulaires ajouterait une complexité inutile. L'adéquation architecture/données
est le premier levier de performance en deep learning, avant même le choix des hyperparamètres.
